## 🚀 Pipeline complet : Parquet → Sliding Window → RNN Version GPU

* Lit ton Dask DataFrame partition par partition.

* Applique la normalisation globale sur chaque partition.

* Crée des séquences glissantes (seq_len, features) pour le RNN.

* Génère des batches prêts pour tf.data.Dataset.

In [1]:
import numpy as np
# Forcer TensorFlow à CPU (si GPU non disponible)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf
import dask.dataframe as dd

# Charger toutes les partitions
df = dd.read_parquet("/home/oem/Documents/these/data/2020/concat_final/dataset_final_2020")

# Forcer time en datetime
df["time"] = dd.to_datetime(df["time"], utc=True)
############################################################
#  🧠 Option 1 : Charger un échantillon (ex. 100 000 lignes)
#df_sample = df_dask.sample(frac=0.01).compute()
##############################################################
# Sélectionner seulement les colonnes utiles pour le RNN
features = ["latitude", "longitude", "level", "U_wind", "V_wind", "Spec_hum","Rel_hum","Temp_2m", "SST"]
target = "Total_precip"  # exemple de variable à prédire
df = df[features + [target]]

seq_len = 24  # longueur de la séquence (ex: 24h)
batch_size = 32

#features = ["Rel_hum", "Spec_hum", "U_wind", "V_wind", "Total_precip", "Temp_2m", "SST"]
#target = "Total_precip"  # exemple de variable à prédire

# -------------------------------
# 1️⃣ Calculer min et max pour features et target
# -------------------------------
f_min = df[features].min().compute()
f_max = df[features].max().compute()

t_min = df[target].min().compute()
t_max = df[target].max().compute()

# -------------------------------
# 2️⃣ Normalisation Min-Max
# -------------------------------
def normalize_partition(pdf, features=features, target=target,
                        f_min=f_min, f_max=f_max, t_min=t_min, t_max=t_max):
    pdf_norm = pdf.copy()

    # Normalisation Min-Max des features
    for col in features:
        pdf_norm[col] = (pdf[col] - f_min[col]) / (f_max[col] - f_min[col] + 1e-8)

    # Normalisation Min-Max du target
    pdf_norm[target] = (pdf[target] - t_min) / (t_max - t_min + 1e-8)

    return pdf_norm


def sliding_window(pdf_partition, seq_len=seq_len, features=features, target=target):
    X, y = [], []
    values = pdf_partition[features].to_numpy(dtype=np.float32)
    targets = pdf_partition[target].to_numpy(dtype=np.float32)
    for i in range(len(values) - seq_len):
        X.append(values[i:i+seq_len])
        y.append(targets[i+seq_len])
    return np.array(X), np.array(y)

def rnn_generator(dask_df, seq_len=seq_len, batch_size=batch_size):
    for partition in dask_df.to_delayed():
        pdf = partition.compute()  # Charger partition en RAM
        pdf = normalize_partition(pdf)  # Normalisation

        X_part, y_part = sliding_window(pdf, seq_len=seq_len)
        
        # Créer des batches
        for i in range(0, len(X_part), batch_size):
            yield X_part[i:i+batch_size], y_part[i:i+batch_size]

# Créer tf.data.Dataset
tf_dataset = tf.data.Dataset.from_generator(
    lambda: rnn_generator(df, seq_len=seq_len, batch_size=batch_size),
    output_signature=(
        tf.TensorSpec(shape=(None, seq_len, len(features)), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32)
    )
).prefetch(tf.data.AUTOTUNE)


2025-08-25 19:34:55.999795: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-25 19:35:03.977258: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-25 19:35:25.677559: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-25 19:42:07.663089: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


1️⃣ Définir le modèle LSTM

In [5]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense

seq_len = 24           # longueur de la séquence (24h)
n_features = len(features)  # nombre de features

model = Sequential([
    LSTM(64, input_shape=(seq_len, n_features), return_sequences=False),
    Dense(1)  # prédiction univariée (Temp_2m)
])

model.compile(optimizer="adam", loss="mse")
model.summary()


/home/oem/anaconda3/envs/climate-env/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,009 (74.25 KB)

 Trainable params: 19,009 (74.25 KB)

 Non-trainable params: 0 (0.00 B)

 2️⃣ Entraînement avec tf_dataset

## 🚀 Pipeline complet : Parquet → Sliding Window → RNN Version CPU

* Forcer TensorFlow en mode CPU

* Normalisation Min-Max (séparée pour features et target)

* Création des séquences temporelles

* Construction et entraînement d’un modèle RNN simple

In [8]:
import os
# 🚫 Désactive CUDA pour forcer TensorFlow à n'utiliser que le CPU
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
import dask.dataframe as dd
import numpy as np

# ============================
# 1. Chargement des données parquet
# ============================
df = dd.read_parquet("/home/oem/Documents/these/data/2020/concat_final/dataset_final_2020")

# Conversion de la colonne time
df["time"] = dd.to_datetime(df["time"], utc=True)
#df = df.set_index("time")

# Sélectionner seulement les colonnes utiles pour le RNN
features = ["latitude", "longitude", "level", "U_wind", "V_wind", "Spec_hum","Rel_hum","Temp_2m", "SST"]
target = "Total_precip"  # exemple de variable à prédire
df = df[features + [target]]

# ============================
# 2. Normalisation Min-Max
# ============================

# Calcul min/max global avec Dask
stats = df[features].describe().compute()
mins  = stats.loc["min"].to_dict()
maxs  = stats.loc["max"].to_dict()

def normalize_partition(pdf, features=features, mins=mins, maxs=maxs, target="Total_precip"):
    pdf_norm = pdf.copy()

    # Normalisation Min-Max pour les features (hors target)
    for col in features:
        if col != target:  # ne pas normaliser la cible ici
            if col in pdf.columns:  # sécurité
                pdf_norm[col] = (pdf[col] - mins[col]) / (maxs[col] - mins[col] + 1e-8)

    # Normalisation séparée pour la cible
    if target in pdf.columns:
        pdf_norm[target] = (pdf[target] - mins[target]) / (maxs[target] - mins[target] + 1e-8)

    return pdf_norm


df_norm = df.map_partitions(normalize_partition)

# ============================
# 3. Conversion vers numpy (batch CPU-friendly)
# ============================
pdf = df_norm.compute()
data = pdf[features].to_numpy(dtype=np.float32)

# ============================
# 4. Création des séquences temporelles
# ============================
seq_length = 24   # séquence de 24h
X, y = [], []

for i in range(len(data) - seq_length):
    X.append(data[i:i+seq_length, :])   # séquence d’entrée
    y.append(data[i+seq_length, features.index("Temp_2m")])  # prédiction Temp_2m

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print("Shape X:", X.shape)  # (nb_samples, 24, nb_features)
print("Shape y:", y.shape)  # (nb_samples,)

# ============================
# 5. Dataset TensorFlow
# ============================
batch_size = 32
tf_dataset = tf.data.Dataset.from_tensor_slices((X, y))
tf_dataset = tf_dataset.shuffle(1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

# ============================
# 6. Modèle RNN
# ============================
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(seq_length, len(features))),
    tf.keras.layers.LSTM(64, return_sequences=False),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1)  # prédiction Temp_2m normalisée
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])

# ============================
# 7. Entraînement sur CPU
# ============================
with tf.device("/CPU:0"):  # 🚨 CPU uniquement
    history = model.fit(tf_dataset, epochs=5)

print("✅ Entraînement terminé sur CPU")


ValueError: Metadata inference failed in `normalize_partition`.

You have supplied a custom function and Dask is unable to 
determine the type of output that that function returns. 

To resolve this please provide a meta= keyword.
The docstring of the Dask function you ran should have more information.

Original error is below:
------------------------
KeyError('Total_precip')

Traceback:
---------
  File "/home/oem/anaconda3/envs/climate-env/lib/python3.10/site-packages/dask/dataframe/utils.py", line 149, in raise_on_meta_error
    yield
  File "/home/oem/anaconda3/envs/climate-env/lib/python3.10/site-packages/dask/dataframe/dask_expr/_expr.py", line 4112, in emulate
    return func(*_extract_meta(args, True), **_extract_meta(kwargs, True))
  File "/tmp/ipykernel_60057/2395298834.py", line 43, in normalize_partition
    pdf_norm[target] = (pdf[target] - mins[target]) / (maxs[target] - mins[target] + 1e-8)


In [3]:
steps_per_epoch = 100  # ajuster selon la taille des partitions / dataset total
epochs = 5

model.fit(
    tf_dataset,
    epochs=epochs,
    steps_per_epoch=steps_per_epoch
)

Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: nan
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: nan
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: nan
Epoch 4/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: nan
Epoch 5/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: nan


## 3️⃣ Optionnel : Multi-année (2020 + 2021) version GPU
Si tu veux entraîner sur 2020 + 2021 :
## Pipeline complet

In [ ]:
import dask.dataframe as dd
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense

# -------------------------------
# 1️⃣ Charger et concaténer les fichiers Parquet
# -------------------------------
df2020 = dd.read_parquet("/home/oem/Documents/these/data/2020/concat_final/dataset_final_2020")
df2021 = dd.read_parquet("/home/oem/Documents/these/data/2021/concat_final/dataset_final_2021")
df = dd.concat([df2020, df2021])

# Conversion de la colonne time en datetime
df["time"] = dd.to_datetime(df["time"], utc=True)

# Colonnes features et target
features = ["Rel_hum", "Spec_hum", "U_wind", "V_wind", "Total_precip", "Temp_2m", "SST"]
target = "Temp_2m"

# -------------------------------
# 2️⃣ Calculer statistiques globales pour normalisation
# -------------------------------
means = df[features].mean().compute()
stds = df[features].std().compute()

def normalize_partition(pdf, features=features, means=means, stds=stds):
    pdf_norm = pdf.copy()
    for col in features:
        pdf_norm[col] = (pdf[col] - means[col]) / stds[col]
    return pdf_norm

# -------------------------------
# 3️⃣ Générateur de séquences RNN
# -------------------------------
seq_len = 24
batch_size = 32

def sliding_window(pdf, seq_len=seq_len, features=features, target=target):
    X, y = [], []
    values = pdf[features].to_numpy(dtype=np.float32)
    targets = pdf[target].to_numpy(dtype=np.float32)
    for i in range(len(values) - seq_len):
        X.append(values[i:i+seq_len])
        y.append(targets[i+seq_len])
    return np.array(X), np.array(y)

def rnn_generator(dask_df, seq_len=seq_len, batch_size=batch_size):
    for partition in dask_df.to_delayed():
        pdf = partition.compute()
        pdf = normalize_partition(pdf)
        X_part, y_part = sliding_window(pdf, seq_len=seq_len)
        for i in range(0, len(X_part), batch_size):
            yield X_part[i:i+batch_size], y_part[i:i+batch_size]

# -------------------------------
# 4️⃣ Créer tf.data.Dataset
# -------------------------------
tf_dataset = tf.data.Dataset.from_generator(
    lambda: rnn_generator(df, seq_len=seq_len, batch_size=batch_size),
    output_signature=(
        tf.TensorSpec(shape=(None, seq_len, len(features)), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32)
    )
).prefetch(tf.data.AUTOTUNE)

# -------------------------------
# 5️⃣ Définir et compiler le modèle LSTM
# -------------------------------
model = Sequential([
    LSTM(64, input_shape=(seq_len, len(features)), return_sequences=False),
    Dense(1)
])
model.compile(optimizer="adam", loss="mse")
model.summary()

# -------------------------------
# 6️⃣ Entraînement
# -------------------------------
steps_per_epoch = 100  # ajuster selon taille dataset
epochs = 5

model.fit(tf_dataset, epochs=epochs, steps_per_epoch=steps_per_epoch)


## 🔹 Étape 3 : Fonctions inverses (denormalisation)

In [ ]:
# Denormaliser les features
def denormalize_features(X_norm, f_min=f_min, f_max=f_max):
    X_denorm = X_norm.copy()
    for col in features:
        X_denorm[col] = X_norm[col] * (f_max[col] - f_min[col] + 1e-8) + f_min[col]
    return X_denorm

# Denormaliser le target
def denormalize_target(y_norm, t_min=t_min, t_max=t_max):
    return y_norm * (t_max - t_min + 1e-8) + t_min


##🔹 Exemple d’utilisation

In [ ]:
# Appliquer la normalisation sur le Dask DataFrame
df_norm = df.map_partitions(normalize_partition)

# Exemple sur Pandas (après compute)
sample = df_norm.head(10)

# Denormalisation
sample_denorm = denormalize_features(sample[features])
sample_target_denorm = denormalize_target(sample[target])
